In [10]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

In [11]:
from langsmith import Client

client = Client()

# Create the dataset
dataset_name = "rag_dataset"
dataset = client.create_dataset(dataset_name=dataset_name)

# Create examples - all must have consistent "inputs" and "outputs" keys
client.create_examples(
    inputs=[
        {"question": "What is the capital of France?"},
        {"question": "What is LangChain?"},
        {"question": "What is LangSmith?"},
        {"question": "What is OpenAI?"},
        {"question": "What is Google?"},
        {"question": "What is Mistral?"},
    ],
    outputs=[
        {"answer": "Paris"},
        {"answer": "A framework for building LLM applications"},
        {"answer": "A platform for observing and evaluating LLM applications"},
        {"answer": "A company that creates Large Language Models"},
        {"answer": "A technology company known for search"},
        {"answer": "A company that creates Large Language Models"},
    ],
    dataset_id=dataset.id,
)

{'example_ids': ['ea0fb0d2-520e-40c8-92e4-32aa8dee4c24',
  '3eae577b-739a-4890-b081-87df5336c069',
  'f4012071-3d8a-4b52-868f-b84f9a8278a8',
  '36be2566-1dc5-4880-818f-748bb47393dc',
  'e4c2ee57-a3b5-4321-8458-791585427286',
  '58ca5000-fc5d-43cc-bf63-08eff3b2d5e5'],
 'count': 6}

Define Metrics and (LLM AS JUDGE)



In [12]:
import os

from google import genai
from google.genai import types
from langsmith import wrappers, Client

from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

# Create Gemini Client (uses GOOGLE_API_KEY from environment)
gemini_client = genai.Client()

# Wrap Gemini Client for LangSmith tracing
google_client = wrappers.wrap_gemini(gemini_client)

# LangSmith Client
ls_client = Client()

eval_instructions = """
You are an expert professor specialized in grading students' answers.

Compare the student's answer with the reference answer.

Respond with ONLY one word:

CORRECT

or

INCORRECT
"""

def correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict,
) -> bool:
    prompt = f"""
Question:
{inputs['question']}

Reference Answer:
{reference_outputs['answer']}

Student Answer:
{outputs['response']}

{eval_instructions}
"""

    response = google_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0,
        ),
    )

    grade = response.text.strip().upper()

    return grade == "CORRECT"


#  +++++++++++++++++++ CONCISIONS++++++++++++



# Concisions-check wheather the actual output is less than 2x the length of expected output




def Concisions(outputs:dict,reference_outputs:dict)->bool:
     if "response" not in outputs:
         return False
     if "answer" not in reference_outputs:
         return False
     return int(len(outputs["response"])) <= 2 * int(len(reference_outputs["answer"]))

NOW ITS time for run the rag system with evaluations

In [13]:
from google import genai
from google.genai import types
import os

google_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))   

default_instruction="Respond to user questions in a short,concise manner (one sentence)"

def my_app(questions:str,model:str= "gemini-2.5-flash",instruction:str=default_instruction)->str:
    response=google_client.models.generate_content(
        model=model,
        contents=questions,
        config=types.GenerateContentConfig(
            system_instruction=instruction,
            temperature=0,
        )
    )
    return response.text


# call my_App for everydata point
def ls_target(input:dict)->dict:
    return {"response": my_app(input["question"])}

run the evaluation

In [14]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")


experiment_result=client.evaluate(
     ls_target,
     data=dataset_name,
     evaluators=[correctness, Concisions],
     experiment_prefix="rag-eval_genai"
)

View the evaluation results for experiment: 'rag-eval_genai-6df2fd17' at:
https://smith.langchain.com/o/acda6f3a-0cfa-4353-910e-f27da82eba70/datasets/326410ac-7d8f-41f2-9a51-8f5d74c9e579/compare?selectedSessions=412636d7-d74a-4a0d-8776-c102971e37ec




3it [00:15,  4.79s/it]Error running target function: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 44.183282666s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location

# Now proper rag with data ingestion  and webased loader

In [ ]:
import os
from dotenv import load_dotenv
!pip install beautifulsoup4
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load environment variables
load_dotenv()

# List of URLs
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
]

# Load documents from URLs
loaders = [WebBaseLoader(url).load() for url in urls]

# Flatten the list of documents
documents_list = [
    document
    for sublist in loaders
    for document in sublist
]

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=200,
    chunk_overlap=0,
)

# Split documents into chunks
split_documents = text_splitter.split_documents(documents_list)

# Initialize Gemini embeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/text-embedding-004"
)

# Create the in-memory vector store
vector_store = InMemoryVectorStore.from_documents(
    documents=split_documents,
    embedding=embeddings,
)

# Create the retriever
retriever = vector_store.as_retriever(
    search_kwargs={"k": 6}
)


retriever.invoke("What is an agent?")

Defaulting to user installation because normal site-packages is not writeable


GoogleGenerativeAIError: Error embedding content (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}